# 03 — Classification / Churn

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("telco_cleaned.csv")
print(df.shape)
display(df.head())

## 2. Define churn target

In [ ]:
if "Churn Label" in df.columns:
    y = df["Churn Label"].map({"Yes": 1, "No": 0})
elif "Churn Value" in df.columns:
    y = pd.to_numeric(df["Churn Value"], errors="coerce")
else:
    raise ValueError("No churn target column found.")

print(y.value_counts(dropna=False))

## 3. Prepare features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

drop_cols = [c for c in ["Churn Label", "Churn Value", "Customer Status", "Customer ID", "Count"] if c in df.columns]
X = df.drop(columns=drop_cols)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 4. Train classification models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
display(results_df)

## 5. Confusion matrix and classification report

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
pred = best_model.predict(X_test)

print(best_name)
print(classification_report(y_test, pred, target_names=["No Churn", "Churn"]))

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 6. Save churn model

In [ ]:
import joblib
joblib.dump(best_model, "churn_model.pkl")
print("Saved: churn_model.pkl")